# LLMs for Text Transformation

In this notebook, we will explore how to use Large Language Models for text transformation tasks such as language translation, spelling and grammar checking, tone adjustment, and format conversion.

## Setup

In [6]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [7]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

## Translation

ChatGPT is trained with sources in many languages. This gives the model the ability to do translation. Here are some examples of how to use this capability.

In [8]:
prompt = f"""
Translate the following English text to Spanish: \ 
```Hi, I would like to order a blender```
"""
response = get_completion(prompt)
print(response)

Hola, me gustaría ordenar una licuadora.


In [9]:
prompt = f"""
Tell me which language this is: 
```Combien coûte le lampadaire?```
"""
response = get_completion(prompt)
print(response)

This is French.


In [10]:
prompt = f"""
Translate the following  text to French and Spanish
and English pirate: \
```I want to order a basketball```
"""
response = get_completion(prompt)
print(response)

French: Je veux commander un ballon de basket
Spanish: Quiero ordenar un balón de baloncesto
English: I want to order a basketball


In [11]:
prompt = f"""
Translate the following text to Spanish in both the \
formal and informal forms: 
'Would you like to order a pillow?'
"""
response = get_completion(prompt)
print(response)

Formal: ¿Le gustaría ordenar una almohada?
Informal: ¿Te gustaría ordenar una almohada?


### Universal Translator
Imagine you are in charge of IT at a large multinational e-commerce company. Users are messaging you with IT issues in all their native languages. Your staff is from all over the world and speaks only their native languages. You need a universal translator!

In [12]:
user_messages = [
  "La performance du système est plus lente que d'habitude.",  # System performance is slower than normal         
  "Mi monitor tiene píxeles que no se iluminan.",              # My monitor has pixels that are not lighting
  "Il mio mouse non funziona",                                 # My mouse is not working
  "Mój klawisz Ctrl jest zepsuty",                             # My keyboard has a broken control key
  "我的屏幕在闪烁"                                               # My screen is flashing
] 

In [13]:
for issue in user_messages:
    prompt = f"Tell me what language this is: ```{issue}```"
    lang = get_completion(prompt)
    print(f"Original message ({lang}): {issue}")

    prompt = f"""
    Translate the following  text to English \
    and Korean: ```{issue}```
    """
    response = get_completion(prompt)
    print(response, "\n")

Original message (French): La performance du système est plus lente que d'habitude.
English: "The system performance is slower than usual."

Korean: "시스템 성능이 평소보다 느립니다." 

Original message (This is Spanish.): Mi monitor tiene píxeles que no se iluminan.
English: "My monitor has pixels that do not light up."

Korean: "내 모니터에는 빛나지 않는 픽셀이 있습니다." 

Original message (Italian): Il mio mouse non funziona
English: My mouse is not working
Korean: 내 마우스가 작동하지 않습니다 

Original message (This is Polish.): Mój klawisz Ctrl jest zepsuty
English: My Ctrl key is broken
Korean: 제 Ctrl 키가 고장 났어요 

Original message (This is Chinese.): 我的屏幕在闪烁
English: My screen is flickering
Korean: 내 화면이 깜박거립니다 



## Tone Transformation
Writing can vary based on the intended audience. ChatGPT can produce different tones.


In [14]:
prompt = f"""
Translate the following from slang to a business letter: 
'Dude, This is Joe, check out this spec on this standing lamp.'
"""
response = get_completion(prompt)
print(response)

Dear Sir/Madam,

I am writing to bring to your attention the specifications of the standing lamp. 

Sincerely,
Joe


## Format Conversion
ChatGPT can translate between formats. The prompt should describe the input and output formats.

In [15]:
data_json = { "resturant employees" :[ 
    {"name":"Shyam", "email":"shyamjaiswal@gmail.com"},
    {"name":"Bob", "email":"bob32@gmail.com"},
    {"name":"Jai", "email":"jai87@gmail.com"}
]}

prompt = f"""
Translate the following python dictionary from JSON to an HTML \
table with column headers and title: {data_json}
"""
response = get_completion(prompt)
print(response)

<html>
<head>
  <title>Restaurant Employees</title>
</head>
<body>
  <table>
    <tr>
      <th>Name</th>
      <th>Email</th>
    </tr>
    <tr>
      <td>Shyam</td>
      <td>shyamjaiswal@gmail.com</td>
    </tr>
    <tr>
      <td>Bob</td>
      <td>bob32@gmail.com</td>
    </tr>
    <tr>
      <td>Jai</td>
      <td>jai87@gmail.com</td>
    </tr>
  </table>
</body>
</html>


In [16]:
from IPython.display import display, Markdown, Latex, HTML, JSON
display(HTML(response))

Name,Email
Shyam,shyamjaiswal@gmail.com
Bob,bob32@gmail.com
Jai,jai87@gmail.com


## Spellcheck/Grammar check.

Here are some examples of common grammar and spelling problems and the LLM's response. 

To signal to the LLM that you want it to proofread your text, you instruct the model to 'proofread' or 'proofread and correct'.

In [17]:
text = [ 
  "The girl with the black and white puppies have a ball.",  # The girl has a ball.
  "Yolanda has her notebook.", # ok
  "Its going to be a long day. Does the car need it’s oil changed?",  # Homonyms
  "Their goes my freedom. There going to bring they’re suitcases.",  # Homonyms
  "Your going to need you’re notebook.",  # Homonyms
  "That medicine effects my ability to sleep. Have you heard of the butterfly affect?", # Homonyms
  "This phrase is to cherck chatGPT for speling abilitty"  # spelling
]
for t in text:
    prompt = f"""Proofread and correct the following text
    and rewrite the corrected version. If you don't find
    and errors, just say "No errors found". Don't use 
    any punctuation around the text:
    ```{t}```"""
    response = get_completion(prompt)
    print(response)

The girl with the black and white puppies has a ball.
No errors found
No errors found
Their goes my freedom. There going to bring they’re suitcases.

No errors found

Rewritten:
Their goes my freedom. There going to bring their suitcases.
You're going to need your notebook.
No errors found.
No errors found


In [18]:
text = f"""
Got this for my daughter for her birthday cuz she keeps taking \
mine from my room.  Yes, adults also like pandas too.  She takes \
it everywhere with her, and it's super soft and cute.  One of the \
ears is a bit lower than the other, and I don't think that was \
designed to be asymmetrical. It's a bit small for what I paid for it \
though. I think there might be other options that are bigger for \
the same price.  It arrived a day earlier than expected, so I got \
to play with it myself before I gave it to my daughter.
"""
prompt = f"proofread and correct this review: ```{text}```"
response = get_completion(prompt)
print(response)

I got this for my daughter for her birthday because she keeps taking mine from my room. Yes, adults also like pandas too. She takes it everywhere with her, and it's super soft and cute. One of the ears is a bit lower than the other, and I don't think that was designed to be asymmetrical. It's a bit small for what I paid for it though. I think there might be other options that are bigger for the same price. It arrived a day earlier than expected, so I got to play with it myself before I gave it to my daughter.


In [19]:
# ! pip3 install redlines

In [20]:
from redlines import Redlines

diff = Redlines(text,response)
display(Markdown(diff.output_markdown))

ModuleNotFoundError: No module named 'redlines'

In [ ]:
prompt = f"""
proofread and correct this review. Make it more compelling. 
Ensure it follows APA style guide and targets an advanced reader. 
Output in markdown format.
Text: ```{text}```
"""
response = get_completion(prompt)
display(Markdown(response))

I purchased this adorable panda plush as a birthday gift for my daughter, as she kept borrowing mine from my room. It's not just for kids - even adults can appreciate the charm of pandas. The plush is incredibly soft and cute, and my daughter carries it everywhere with her. However, I did notice that one of the ears is slightly lower than the other, which seems like a manufacturing flaw rather than intentional design. While I found the plush to be a bit smaller than expected given the price, I believe there are larger options available for the same cost. Despite this, the plush arrived a day earlier than anticipated, allowing me to enjoy it myself before passing it on to my daughter. Overall, it's a delightful gift that brings joy to both children and adults alike.

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [21]:
text = [ 
  "The girl with the black and white puppies have a ball.",  # The girl has a ball.
  "Yolanda has her notebook.", # ok
  "Its going to be a long day. Does the car need it’s oil changed?",  # Homonyms
  "Their goes my freedom. There going to bring they’re suitcases.",  # Homonyms
  "Your going to need you’re notebook.",  # Homonyms
  "That medicine effects my ability to sleep. Have you heard of the butterfly affect?", # Homonyms
  "This phrase is to cherck chatGPT for speling abilitty"  # spelling
]

In [26]:
# Prompt 1
for t in text:
    prompt = f"""Proofread and correct the following text.\
    Then, compose a rhyme inspired on the text. Don't use 
    any punctuation around the text:
    ```{t}```"""
    response = get_completion(prompt)
    print(response)

The girl with the black and white puppies have a ball

The girl with the black and white puppies, having a ball
Playing and running, having a ball
Their fur so soft, their tails so tall
The girl with the black and white puppies, having a ball
Yolanda has her notebook

In her hand, it she took

Writing down thoughts that shook

Her world, like a brook.
Its going to be a long day Does the car need its oil changed

In the morning light, I see the way
A long day ahead, no time to be deranged
Does the car need its oil changed
I hope not, for that would cause dismay
Their goes my freedom there going to bring they’re suitcases

Their freedom is gone, they're packing their cases
Leaving behind what once was their oasis
Your going to need your notebook

In a cozy little nook
To jot down all your thoughts
And all the lessons you've been taught
That medicine affects my ability to sleep have you heard of the butterfly effect

In the night, I toss and turn,
That medicine makes my mind churn,
Butter

In [24]:
# Prompt 2
for t in text:
    prompt = f"""Proofread and correct the following text.\
    Then, convert each letter into the ordinal number according to the alphabet. Don't use 
    any punctuation around the text:
    ```{t}```"""
    response = get_completion(prompt)
    print(response)

20 8 5 7 9 18 12 23 9 20 8 20 8 5 2 12 1 3 11 1 14 4 23 9 20 8 20 8 5 2 12 1 3 11 1 14 4 23 9 20 8 16 21 16 16 9 5 19 8 1 22 5 1 2 1 12 12
25 15 12 1 14 4 1 8 1 19 8 5 18 14 15 20 5 2 15 15 11
9 20 19 7 15 9 14 7 20 15 2 5 1 12 15 14 7 4 1 25 4 15 5 19 20 15 2 5 1 12 15 14 7 4 1 25 4 15 5 19 20 8 5 3 1 18 14 5 5 4 9 20 19 15 9 12 3 8 1 14 7 5 4 4 15 5 19 20 9 20 19 15 9 12 3 8 1 18 14 7 5 4
20 8 5 9 18 7 15 5 19 13 25 6 18 5 5 4 15 13. 20 8 5 18 5 7 15 9 14 7 20 15 2 18 9 14 7 20 8 5 9 18 5 19 21 9 20 3 1 19 5 19.
25 15 21 18 7 15 9 14 7 20 15 14 5 5 4 25 15 21 18 5 5 14 5 5 4 25 15 21 18 5 14 15 20 5 2 15 15 11
20 8 1 20 13 5 4 9 3 9 14 5  5 6 6 5 3 20 19  13 25  1 2 9 12 9 20 25  20 15  19 12 5 5 16
20 8 9 19 16 18 1 19 5 9 19 20 15 3 8 5 18 3 11 3 8 1 20 7 16 20 6 15 18 19 16 5 12 9 14 7 1 2 9 12 9 20 20 25


In [27]:
# Prompt 3
for t in text:
    prompt = f"""Proofread and correct the following text.\
    Then, rewrite the text forcing the verb into the simple future tense. Don't use 
    any punctuation around the text:
    ```{t}```"""
    response = get_completion(prompt)
    print(response)

The girl with the black and white puppies has a ball

The girl with the black and white puppies will have a ball
Yolanda has her notebook.

Yolanda will have her notebook.
Its going to be a long day does the car need its oil changed

It will be a long day will the car need its oil changed
Their goes my freedom there going to bring they’re suitcases

Their will go my freedom their will go to bring they’re suitcases
Your going to need you’re notebook.

You will need your notebook.
Then rewrite the text forcing the verb into the simple future tense: 

That medicine will affect my ability to sleep have you heard of the butterfly effect
This phrase is to cherck chatGPT for speling abilitty

This phrase will check chatGPT for spelling ability


Comment:
- Prompt 1 (Correct + Rhyme): GPT handled corrections well, but rhyme quality degraded for short inputs.
- Prompt 2 (Correct + Alphabet Ordinals): This is where the clearest failures appeared. GPT produced wrong number sequences for longer sentences. A form of arithmetic hallucination, confidently returning incorrect character-to-number mappings. This is the most interesting finding to discuss.
- Prompt 3 (Correct + Future Tense): Performed the best overall. Tense transformation is a natural linguistic task for LLMs.